# PROB IP102 Continual Learning Evaluation & Training Pipeline

Notebook này cung cấp quy trình (pipeline) hoàn chỉnh để chạy đánh giá và huấn luyện mô hình **PROB** (Probabilistic Open-World Object Detection) với bộ dữ liệu sâu bệnh **IP102** trên **Kaggle** sử dụng **2 GPU T4 (DDP)**.

### Hướng dẫn chuẩn bị trước khi chạy:
1. Đảm bảo Notebook này được cấu hình chạy trên **Accelerator: GPU T4 x2** (ở panel Settings phía bên phải).
2. Đảm bảo **Internet** được bật (Internet on).
3. Thêm các Dataset sau vào Notebook:
   - Tập dữ liệu ảnh IP102 (đường dẫn: `/kaggle/input/datasets/rtlmhjbn/ip02-dataset`).
   - Tập nhãn COCO của IP102 (đường dẫn: `/kaggle/input/datasets/eljazouly/ip102-coco-annotations`).
   - Thư mục chứa các checkpoint pretrain tác giả hoặc của bạn (ví dụ: `prob-checkpoints` chứa `t1.pth`, `t2.pth`, `t3.pth`, `t4.pth`).

## Bước 1: Clone Repository và Biên dịch thư viện CUDA

In [ ]:
# Clone repo chứa mã nguồn tùy chỉnh đã được cấu hình lớp IP102
!if [ ! -d Prob-ann-custom ]; then git clone https://github.com/nta2112/Prob-ann-custom.git; else cd Prob-ann-custom && git pull; fi
%cd Prob-ann-custom

In [ ]:
# Cài đặt các thư viện yêu cầu
!pip install -r requirements.txt
!pip install pycocotools einops

# Biên dịch các toán tử CUDA MultiScaleDeformableAttention
%cd models/ops
!python setup.py build install
%cd ../..

## Bước 2: Thiết lập dữ liệu IP102 (Chuyển đổi sang định dạng VOC XML)

Mã nguồn PROB sử dụng dataloader định dạng VOC XML. Lệnh dưới đây chạy script `setup_ip102.py` để quét ảnh, tự động vá file cấu hình lớp `open_world.py` và chuyển nhãn từ COCO JSON sang VOC XML.

In [ ]:
# Cấu hình đường dẫn dữ liệu chuẩn của IP102 trên Kaggle
JSON_DIR = "/kaggle/input/datasets/eljazouly/ip102-coco-annotations/coco_annotations"
IMAGE_DIR = "/kaggle/input/datasets/rtlmhjbn/ip02-dataset"
OUTPUT_DIR = "/kaggle/working/datasets/IP102"

!python tools/setup_ip102.py \
    --json_dir {JSON_DIR} \
    --image_dir {IMAGE_DIR} \
    --output_dir {OUTPUT_DIR}

## Bước 3: Chạy huấn luyện thử nghiệm 1 Epoch & Đánh giá xen kẽ cho từng Task (Task 1 -> Task 4)

Lệnh dưới đây chạy kịch bản `run_train_eval_kaggle.sh` để thực hiện **huấn luyện tinh chỉnh 1 Epoch rồi chạy đánh giá (test) ngay lập tức** cho từng task liên tiếp:
- Tham số thứ nhất: Đường dẫn thư mục chứa checkpoint ban đầu (`/kaggle/input/prob-checkpoints`)
- Tham số thứ hai: Batch size huấn luyện trên mỗi GPU (Khuyến nghị: `4` để tránh OOM)
- Tham số thứ ba: Batch size đánh giá trên mỗi GPU (Khuyến nghị: `16` để chạy nhanh hơn)

**Quy trình thực thi:**
1. **Task 1:** Huấn luyện 1 Epoch (từ checkpoint pretrain `t1.pth`) -> Đánh giá kiểm thử ngay lập tức trên lớp 1-27.
2. **Task 2:** Huấn luyện 1 Epoch (từ checkpoint của Task 1) -> Tinh chỉnh mẫu 1 Epoch -> Đánh giá kiểm thử ngay lập tức trên lớp 1-52.
3. **Task 3:** Huấn luyện 1 Epoch (từ checkpoint của Task 2) -> Tinh chỉnh mẫu 1 Epoch -> Đánh giá kiểm thử ngay lập tức trên lớp 1-77.
4. **Task 4:** Huấn luyện 1 Epoch (từ checkpoint của Task 3) -> Tinh chỉnh mẫu 1 Epoch -> Đánh giá kiểm thử ngay lập tức trên lớp 1-102.

In [ ]:
# Chạy quy trình xen kẽ Huấn luyện 1 Epoch -> Đánh giá kiểm thử cho từng Task
!bash tools/run_train_eval_kaggle.sh /kaggle/input/prob-checkpoints 4 16

## Bước 4: Chạy Đánh giá trực tiếp (Nếu checkpoint đã tương thích sẵn)

Chỉ sử dụng lệnh này nếu Sans đã có sẵn các checkpoint đã được huấn luyện sẵn trên IP102 (đầy đủ 103 lớp đầu ra). Lệnh này sẽ bỏ qua huấn luyện và chạy đánh giá test trực tiếp trên cả 4 task bằng 2 GPU.

In [ ]:
!bash tools/run_eval_kaggle.sh /kaggle/input/prob-checkpoints 16